In [1]:

import numpy as np
from IPython.display import IFrame

In [2]:
import requests

In [3]:
# Start the Visor server on the command line using:
# visor-cli server start

# Take note of the service endpoint and assign to the url variable here.
url = "http://localhost:53211"

# Set up endpoints for the operations we will need
# Start
start_url = f"{url}/start"
# List datasets
list_datasets_url = f"{url}/list_datasets"
# List variables (for dataset)
def list_variables_url(dataset_id):
    return f"{url}/{dataset_id}/list_variables"
# Update variables (for dataset)
def update_variables_url(dataset_id):
    return f"{url}/{dataset_id}/update_variables"

In [4]:
# Set up the start url and payload
start_payload={"file_path": "examples/assets/vtk_scene_sphere_l2_b3_r32_v3_c1_z0.vtm",
               "metadata": "examples/assets/vtk_scene_sphere_l2_b3_r32_v3_c1_z0.json"}

# Submit the request
resp = requests.post(start_url, json=start_payload)
resp.content
dataset_id1 = resp.json().get("dataset_id")


In [5]:
IFrame("http://localhost:8081", width="1000", height="500")

In [6]:
# Select the parent dataset in the tree view, and color by Normals

In [7]:
# List datasets and get the dataset ID
resp = requests.get(list_datasets_url)
datasets = resp.json().get('datasets', {})
datasets

{'2278968957650919': {'id': 2278968957650919,
  'name': 'vtk_scene_sphere_l2_b3_r32_v3_c1_z0',
  'unit': 'm',
  'file_path': 'examples/assets/vtk_scene_sphere_l2_b3_r32_v3_c1_z0.vtm',
  'metadata_path': 'examples/assets/vtk_scene_sphere_l2_b3_r32_v3_c1_z0.json'}}

In [8]:
# List variables for dataset_id
resp = requests.get(list_variables_url(dataset_id1))
parts = resp.json().get('parts')
# Choose the last part to update
part_to_update = parts[-1]
part_id = part_to_update.get("part_id")
variables = part_to_update.get("variables")
part_to_update

{'part_id': 3554123176990211,
 'part_name': 'level_1_block_2_part_2',
 'variables': [{'index': 0,
   'type': 'POINT',
   'name': 'Normals',
   'num_components': 3,
   'num_points': 962,
   'ranges': [[-0.9987165331840515, 0.9987165331840515],
    [-0.9987165331840515, 0.9987165331840515],
    [-1.0, 1.0]],
   'magnitude_range': [0.9999999680464912, 1.0000000418852286]},
  {'index': 1,
   'type': 'POINT',
   'name': 'gradient_variable_0',
   'num_components': 1,
   'num_points': 962,
   'ranges': [[0.0, 1.0]],
   'magnitude_range': [0.0, 1.0]},
  {'index': 2,
   'type': 'POINT',
   'name': 'constant_variable_1',
   'num_components': 1,
   'num_points': 962,
   'ranges': [[1.0, 1.0]],
   'magnitude_range': [1.0, 1.0]},
  {'index': 3,
   'type': 'POINT',
   'name': 'random_variable_2',
   'num_components': 1,
   'num_points': 962,
   'ranges': [[0.0001607129815965891, 0.9997527003288269]],
   'magnitude_range': [0.0001607129815965891, 0.9997527003288269]}]}

In [9]:
# Create the new variables for the test point dataArray (zeros):
var_to_update = variables[0]
name = var_to_update["name"]
var_type = var_to_update["type"]
num_points = var_to_update["num_points"]
num_components = var_to_update["num_components"]

print(f"Updating part {part_to_update.get('part_name')}, variable {name}, {var_type}, {num_components}")

new_vector_values = np.zeros((num_points, num_components))
# need to flatten to a python list of num_points*num_components values
new_vector_values = [0]*num_points*num_components

Updating part level_1_block_2_part_2, variable Normals, POINT, 3


In [10]:
# Create new vector data
new_vector_values = np.zeros((num_points, num_components))
# need to flatten to a python list of num_points*num_components values
new_vector_values = new_vector_values.flatten().tolist()

In [11]:
# Compile a dict with the vector variable metadtata + updated values
vector_update_info = {
    "type": "point",
    "name": name,
    "num_components": num_components,
    "data": new_vector_values,
    "part_id": part_id
}

In [12]:
# Run the actual update command
payload = {"variables": [vector_update_info]}
resp = requests.post(update_variables_url(dataset_id1), json=payload)